In [17]:
import pandas as pd
import numpy as np

# 1. Load the original dataset from your Mac Downloads folder
file_path = '/Users/macbookpro/Downloads/cumulatief-overzicht-du-vergunningen-2004-augustus-2026.csv'
df_orig = pd.read_csv(file_path, encoding='latin-1', sep=';')
df_orig.columns = [c.strip() for c in df_orig.columns]

# 2. Strict European Waarde Parser
def parse_waarde_strict(val):
    if pd.isna(val):
        return 0.0
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).replace('\x80', '').replace('€', '').strip()
    if not s:
        return 0.0
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    elif '.' in s:
        s = s.replace('.', '')
    try:
        return float(s)
    except ValueError:
        return 0.0

df_orig['Waarde_num'] = df_orig['Waarde'].apply(parse_waarde_strict)

# 3. Filter to the 372 Clearly Relevant Semiconductor Records
def classify_relevance(row):
    desc = str(row['Omschrijving goederen']).lower()
    code = str(row['SG-Post']).lower()
    
    if any(term in desc for term in ['gasmasker', 'gasmaskers', 'urani', 'bestralings', 'cbrn']):
        return 'Not relevant'
    if any(c in code for c in ['0c002', '1a004', '4a506']):
        return 'Not relevant'
        
    clearly_relevant_terms = [
        'lithografische apparatuur', 'lithografie', 'waferstepper', 'wafer stepper', 'wafer scanner',
        'productie-apparatuur voor halfgeleiderelementen', 'productieapparatuur voor halfgeleiderelementen',
        'mocvd', 'epitaxial', 'epitaxiale', 'waferfabr', 'wafer processing',
        'delen voor lithografische apparatuur', 'deel voor lithografische apparatuur',
        'delen voor mocvd', 'delen voor epitaxiale', 'delen van lasermodules voor wafersteppers',
        'delen voor wafersteppers', 'onderdelen voor lithografische apparatuur'
    ]
    
    for term in clearly_relevant_terms:
        if term in desc:
            return 'Clearly relevant'
            
    if ('3b001' in code or '3b002' in code) and any(w in desc for w in ['halfgeleider', 'semiconductor', 'wafer', 'reactor', 'chemische damp', 'cvd', 'pvd', 'etch', 'ets', 'implant', 'masker', 'mask-making']):
        return 'Clearly relevant'

    return 'Not relevant'

df_orig['Relevance_Category'] = df_orig.apply(classify_relevance, axis=1)
clearly_rel = df_orig[df_orig['Relevance_Category'] == 'Clearly relevant'].copy()

print(f"=== SEGMENT 1 COMPLETE ===")
print(f"Total verified records loaded: {len(clearly_rel)}")
print(f"Unique licence numbers: {clearly_rel['Nummer'].nunique()}")
overall_total = clearly_rel['Waarde_num'].sum()
print(f"Definitive Total Authorised Value: €{overall_total:,.2f}")

=== SEGMENT 1 COMPLETE ===
Total verified records loaded: 379
Unique licence numbers: 379
Definitive Total Authorised Value: €117,509,569,323.00


In [19]:
# ==========================================
# SEGMENT 2: DEFINITIVE CATEGORY FINANCIAL AUDIT
# ==========================================
def classify_record_definitively(row):
    code = str(row['SG-Post']).upper()
    desc = str(row['Omschrijving goederen']).lower()
    
    has_explicit_national = any(k in code for k in ['3B801', '3B901', '3B903', '3B904', '3D805', '3D806', '3D901', '3D902', '3D907', '3E901', '3E905'])
    is_mixed = ';' in code and any(c in code for c in ['3B001', '3B002', '3B801', '3B901', '3B']) and any(c in code for c in ['5A', '6A', '4A', '5D'])
    generic_terms = ['diverse goederen', 'onderdelen', 'delen', 'software en technologie']
    is_generic = all(g in desc for g in generic_terms) or len(desc.strip()) < 5 or desc in ['diverse goederen', 'onderdelen']
    
    if has_explicit_national:
        return 'Direct national-code match (unverified thresholds)'
    elif is_mixed:
        return 'Mixed authorisation'
    elif is_generic:
        return 'Other / insufficient information'
    elif any(c in code for c in ['3B001', '3B002']) and any(w in desc for w in ['litho', 'waferstep', 'scanner', 'epitaxi', 'mocvd', 'etch', 'cvd', 'ald', 'sem']):
        return 'Possible match (equipment description present, technical specs unverified)'
    elif any(c in code for c in ['3C002', '3A001', '3D', '3E001']) and not any(c in code for c in ['3B001', '3B002']):
        return 'EU Annex I control only'
    else:
        return 'Other / insufficient information'

clearly_rel['Definitive_Category'] = clearly_rel.apply(classify_record_definitively, axis=1)

# Verify every record has exactly one category
category_counts_per_record = clearly_rel.groupby('Nummer')['Definitive_Category'].count()
assert (category_counts_per_record == 1).all(), "Error: Some records have multiple or zero categories."

# Aggregate and assert equality
cat_summary = clearly_rel.groupby('Definitive_Category').agg(
    Record_Count=('Nummer', 'count'),
    Total_Authorised_Value=('Waarde_num', 'sum')
).reset_index()

cat_summary['Record_Share (%)'] = (cat_summary['Record_Count'] / len(clearly_rel)) * 100
cat_summary['Value_Share (%)'] = (cat_summary['Total_Authorised_Value'] / overall_total) * 100

print("=== SEGMENT 2 COMPLETE: DEFINITIVE CATEGORY BREAKDOWN ===")
print(cat_summary.to_string(index=False))

sum_cat_values = cat_summary['Total_Authorised_Value'].sum()
assert np.isclose(sum_cat_values, overall_total), f"Discrepancy: {overall_total - sum_cat_values}"
print(f"\nAssertion Passed: Category totals sum exactly to €{overall_total:,.2f} with zero discrepancy.")

=== SEGMENT 2 COMPLETE: DEFINITIVE CATEGORY BREAKDOWN ===
                                                       Definitive_Category  Record_Count  Total_Authorised_Value  Record_Share (%)  Value_Share (%)
                                                   EU Annex I control only             6            1.350000e+06          1.583113         0.001149
                                                       Mixed authorisation            61            1.161622e+11         16.094987        98.853420
                                          Other / insufficient information            55            7.672660e+08         14.511873         0.652939
Possible match (equipment description present, technical specs unverified)           257            5.787256e+08         67.810026         0.492492

Assertion Passed: Category totals sum exactly to €117,509,569,323.00 with zero discrepancy.


In [23]:
# ==========================================
# SEGMENT 3: DESTINATION SUMMARY (TRANSIT ISOLATED)
# ==========================================
clearly_rel['Dest_Normalized'] = clearly_rel['Land van eindbestemming'].fillna('MISSING').astype(str).str.strip().str.upper()

def assign_destination_type_transit_fix(dest):
    if dest == 'MISSING' or not dest:
        return 'Missing / Unknown'
    aggregate_labels = [
        'DIVERSE', 'DIVERSE LANDEN', 'WERELD MUV GEVOELIGE LANDEN', 
        'ALLE LANDEN (M.U.V. EMBARGOLANDEN)', 'VERSCHILLENDE LANDEN',
        'EUROPESE UNIE', 'EU, NAVO+', 'DIVERSEN'
    ]
    if dest in aggregate_labels or any(agg in dest for agg in ['DIVERSE', 'WERELD', 'ALLE LANDEN', 'VERSCHILLENDE']):
        return 'Aggregate Destination'
    if ' VIA ' in dest or ' VIA' in dest or 'VIA ' in dest:
        return 'Transit / Routing'
    if ',' in dest or ';' in dest or '/' in dest or ' EN ' in dest or ' AND ' in dest:
        return 'Combined-Country'
    return 'Individual Country'

clearly_rel['Destination_Type_Transit'] = clearly_rel['Dest_Normalized'].apply(assign_destination_type_transit_fix)

dest_summary = clearly_rel.groupby('Destination_Type_Transit').agg(
    Record_Count=('Nummer', 'count'),
    Total_Authorised_Value=('Waarde_num', 'sum')
).reset_index()

dest_summary['Record_Share (%)'] = (dest_summary['Record_Count'] / len(clearly_rel)) * 100
dest_summary['Value_Share (%)'] = (dest_summary['Total_Authorised_Value'] / overall_total) * 100

print("=== SEGMENT 3 COMPLETE: RECALCULATED DESTINATION SUMMARY ===")
print(dest_summary.to_string(index=False))

# Dynamic assertion based on actual loaded rows
expected_count = len(clearly_rel)
assert dest_summary['Record_Count'].sum() == expected_count, f"Expected {expected_count}, got {dest_summary['Record_Count'].sum()}"
assert np.isclose(dest_summary['Total_Authorised_Value'].sum(), overall_total)
print(f"\nAssertion Passed: Destination summary sums to exactly {expected_count} records and full value.")

=== SEGMENT 3 COMPLETE: RECALCULATED DESTINATION SUMMARY ===
Destination_Type_Transit  Record_Count  Total_Authorised_Value  Record_Share (%)  Value_Share (%)
   Aggregate Destination            41            5.571104e+10         10.817942        47.409794
        Combined-Country            52            1.252006e+10         13.720317        10.654499
      Individual Country           265            4.923779e+10         69.920844        41.901089
       Transit / Routing            21            4.068048e+07          5.540897         0.034619

Assertion Passed: Destination summary sums to exactly 379 records and full value.
